# Comparaison des modèles et décisions de modélisation

Ce notebook justifie le modèle déployé par `src/train.py` : du plus simple au plus complexe, chaque candidat est
évalué de la même façon, sur la métrique qui compte pour l'atelier.

## Une métrique événementielle

Le système évalue chaque machine toutes les 30 minutes. Compter les fausses alertes par pas pénaliserait plusieurs
fois le même dérangement : une machine en alerte 10 h d'affilée, c'est une seule visite du technicien. On suit donc :

- le **recall événementiel** : part des pannes précédées d'au moins une alerte dans les 120 h ;
- les **fausses alertes par machine et par jour**, comptées en épisodes (une suite d'alertes = une intervention ;
  au-delà de 24 h, l'épisode est redécoupé pour qu'un modèle bloqué en alerte ne passe pas pour parfait).

Le seuil de chaque modèle est celui qui maximise le recall sous un budget de **0,2 fausse alerte par machine et par
jour** (un dérangement inutile tous les 5 jours par machine).

## Protocole

- Trois fenêtres de validation (T2, T3 et T4 2024), chacune entraînée sur ce qui précède, avec un tampon de 120 h
  pour que la cible ne déborde pas sur la période suivante.
- Test sur 2025, avec le seuil choisi sur la dernière fenêtre, appliqué tel quel.
- Une machine sur cinq n'est jamais vue à l'entraînement : elle mesure la généralisation à une machine neuve.

In [1]:
import sys
import time

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src.config import CIBLE
from src.donnees import PLIS, charger, decouper, decouper_pli
from src.evaluation import (
    BUDGET_FAUSSES_ALERTES_PAR_MACHINE_JOUR,
    PAS_HORIZON,
    grille_seuils,
    metriques_evenementielles,
    trouver_seuil_pour_budget_fa,
)
from src.features import FEATURES, preparer_X
from src.train import nouveau_modele

df = charger()
_, _, test = decouper(df)

## Les candidats, du plus simple au plus complexe

In [2]:
class RegleSeuil(ClassifierMixin, BaseEstimator):
    """Score = les heures d'outil. Aucun apprentissage : le point de comparaison minimal."""

    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        score = X["heures_depuis_changement_outil"].fillna(0).to_numpy(dtype=float)
        return np.column_stack([-score, score])


numeriques = [f for f in FEATURES if f != "type_machine"]
encodeur = lambda numerique: make_column_transformer(
    (numerique, numeriques), (OneHotEncoder(handle_unknown="ignore"), ["type_machine"])
)
CANDIDATS = {
    "Règle : heures d'outil": RegleSeuil,
    "Régression logistique": lambda: make_pipeline(
        encodeur(make_pipeline(SimpleImputer(strategy="median"), StandardScaler())),
        LogisticRegression(max_iter=3000, class_weight="balanced"),
    ),
    "Arbre de décision": lambda: make_pipeline(
        encodeur("passthrough"), DecisionTreeClassifier(max_depth=6, min_samples_leaf=500, random_state=42)
    ),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.1, max_leaf_nodes=15, min_samples_leaf=80, categorical_features="from_dtype", random_state=42
    ),
    "LightGBM (déployé)": nouveau_modele,
}

In [3]:
def recall_au_budget(fenetre, scores):
    return trouver_seuil_pour_budget_fa(
        fenetre, scores, grille_seuils(scores), PAS_HORIZON, BUDGET_FAUSSES_ALERTES_PAR_MACHINE_JOUR
    )


def evaluer(fabrique):
    ligne, pannes_validation = {}, 0
    for i, (debut, fin) in enumerate(PLIS, start=1):
        train, validation = decouper_pli(df, debut, fin)
        depart = time.perf_counter()
        modele = fabrique().fit(preparer_X(train), train[CIBLE])
        ligne["entrainement_s"] = time.perf_counter() - depart
        seuil, m = recall_au_budget(validation, modele.predict_proba(preparer_X(validation))[:, 1])
        ligne[f"recall_T{i + 1}_2024"] = m["recall_evenementiel"]
        pannes_validation += m["nb_pannes"]
    # Protocole de déploiement : le seuil de la dernière fenêtre, appliqué tel quel au test 2025
    m = metriques_evenementielles(test, modele.predict_proba(preparer_X(test))[:, 1], seuil, PAS_HORIZON)
    recall_validation = np.mean([ligne[f"recall_T{i + 1}_2024"] for i in range(1, len(PLIS) + 1)])
    return ligne | {
        "recall_validation": recall_validation,
        # Écart-type binomial : sous ~2 écarts-types, deux modèles ne sont pas départagés
        "ecart_type_validation": np.sqrt(recall_validation * (1 - recall_validation) / pannes_validation),
        "recall_test_2025": m["recall_evenementiel"],
        "ecart_type_test": np.sqrt(m["recall_evenementiel"] * (1 - m["recall_evenementiel"]) / m["nb_pannes"]),
        "fa_test": m["taux_fausses_alertes_par_machine_jour"],
    }


resultats = pd.DataFrame({nom: evaluer(fabrique) for nom, fabrique in CANDIDATS.items()}).T
colonnes = ["recall_validation", "ecart_type_validation", "recall_test_2025", "ecart_type_test", "fa_test", "entrainement_s"]
resultats[colonnes].astype(float).round(3)

,recall_validation,ecart_type_validation,recall_test_2025,ecart_type_test,fa_test,entrainement_s
Règle : heures d'outil,0.593,0.061,0.496,0.047,0.210,0.145
Régression logistique,0.583,0.061,0.278,0.042,0.117,4.304
Arbre de décision,0.527,0.062,0.478,0.047,0.231,8.221
HistGradientBoosting,0.689,0.057,0.530,0.047,0.194,8.345
LightGBM (déployé),0.705,0.057,0.600,0.046,0.199,3.521


## Lecture

Le boosting bat nettement la règle, la régression logistique et l'arbre. Entre les deux boostings, l'écart reste
dans l'incertitude (quelques dizaines de pannes par fenêtre) : LightGBM est conservé, plus rapide et plus léger.
Départager plus finement demanderait plus de pannes, donc plus de machines simulées.

## Ce qui compte pour le modèle

In [4]:
train, _, _ = decouper(df)
lgbm = nouveau_modele().fit(preparer_X(train), train[CIBLE])
pd.Series(lgbm.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)

heures_depuis_changement_outil         535
heures_depuis_maintenance              485
vibration_max_24h                      200
concentration_poussiere_max_24h        143
ecart_thermique_max_24h                114
temperature_process_max_24h            111
type_machine                            91
couple_max_24h                          91
concentration_poussiere_moyenne_24h     71
vibration_moyenne_24h                   67
dtype: int32

## Décisions de modélisation

- **Pas de standardisation, catégorie native** : un modèle à base d'arbres est insensible aux transformations
  monotones ; LightGBM gère les valeurs manquantes et la gamme (`type_machine`) nativement. Les catégories sont
  fixées (L, M, H) pour qu'une machine prédite seule soit lue comme à l'entraînement.
- **Calibration sigmoïde sur les log-odds** : elle rend les probabilités interprétables (sur 100 machines à 10 %,
  une dizaine tombe en panne dans les 5 jours). Ajustée sur la probabilité brute, bornée à 0, elle imposait un
  plancher d'environ 2,8 % à toutes les machines saines ; sur les log-odds, ce plancher disparaît, au même recall.
- **OSF (surcharge mécanique) reste le point faible**, faute d'exemples. Trois pistes testées puis écartées : un
  classifieur dédié (moins bon que le modèle commun), des horizons plus courts pour HDF et OSF (moins d'exemples
  positifs), et une surpondération des pannes OSF (neutre au mieux).
- **50 machines** : un essai à 100 machines déplaçait les résultats par mode, y compris pour la règle qui n'apprend
  rien, signe d'un effet d'échantillon et non d'architecture.